# S2.3 — Error Handling & Middleware

Verifies:
1. Custom exception hierarchy (15+ classes)
2. Structured JSON error responses (`ErrorResponse` schema)
3. Global exception handlers (404, 422, 429, 503, 500)
4. Request logging middleware (X-Request-ID, timing, exclusions)
5. Debug mode traceback inclusion

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "../.."))
os.environ.setdefault("POSTGRES__HOST", "localhost")
os.environ.setdefault("POSTGRES__PORT", "5432")
os.environ.setdefault("POSTGRES__USER", "paper")
os.environ.setdefault("POSTGRES__PASSWORD", "paper")
os.environ.setdefault("POSTGRES__DB", "paperalchemy")
print("Path configured")

Path configured


## 1. Exception Hierarchy

In [2]:
from src.exceptions import (
    PaperAlchemyError, RepositoryError, PaperNotFoundError, PaperSaveError,
    ParsingError, PDFParsingError, PDFValidationError,
    ExternalServiceError, ArxivAPIError, ArxivTimeoutError, ArxivRateLimitError,
    EmbeddingServiceError, LLMServiceError, LLMConnectionError, LLMTimeoutError,
    SearchEngineError, CacheServiceError, PipelineError, ConfigurationError,
)

# Test base error
err = PaperAlchemyError("test error", status_code=418, context={"key": "val"})
assert err.detail == "test error"
assert err.status_code == 418
assert err.context == {"key": "val"}
assert str(err) == "test error"

# Test inheritance chain
assert issubclass(PaperNotFoundError, RepositoryError)
assert issubclass(RepositoryError, PaperAlchemyError)
assert issubclass(ArxivRateLimitError, ArxivAPIError)
assert issubclass(ArxivAPIError, ExternalServiceError)
assert issubclass(ExternalServiceError, PaperAlchemyError)
assert issubclass(LLMConnectionError, LLMServiceError)
assert issubclass(PDFValidationError, PDFParsingError)
assert issubclass(PDFParsingError, ParsingError)

# Test default status codes
assert PaperNotFoundError().status_code == 404
assert PDFValidationError().status_code == 422
assert ArxivRateLimitError().status_code == 429
assert ExternalServiceError().status_code == 503
assert PaperAlchemyError().status_code == 500

print("\n".join([
    f"  PaperAlchemyError (500)",
    f"  ├── RepositoryError",
    f"  │   ├── PaperNotFoundError (404)",
    f"  │   └── PaperSaveError (500)",
    f"  ├── ParsingError",
    f"  │   ├── PDFParsingError",
    f"  │   └── PDFValidationError (422)",
    f"  ├── ExternalServiceError (503)",
    f"  │   ├── ArxivAPIError",
    f"  │   │   ├── ArxivTimeoutError",
    f"  │   │   └── ArxivRateLimitError (429)",
    f"  │   ├── EmbeddingServiceError",
    f"  │   ├── LLMServiceError",
    f"  │   │   ├── LLMConnectionError",
    f"  │   │   └── LLMTimeoutError",
    f"  │   ├── SearchEngineError",
    f"  │   └── CacheServiceError",
    f"  ├── PipelineError",
    f"  └── ConfigurationError",
]))
print("\n✓ Exception hierarchy verified (15 classes, correct inheritance + status codes)")

  PaperAlchemyError (500)
  ├── RepositoryError
  │   ├── PaperNotFoundError (404)
  │   └── PaperSaveError (500)
  ├── ParsingError
  │   ├── PDFParsingError
  │   └── PDFValidationError (422)
  ├── ExternalServiceError (503)
  │   ├── ArxivAPIError
  │   │   ├── ArxivTimeoutError
  │   │   └── ArxivRateLimitError (429)
  │   ├── EmbeddingServiceError
  │   ├── LLMServiceError
  │   │   ├── LLMConnectionError
  │   │   └── LLMTimeoutError
  │   ├── SearchEngineError
  │   └── CacheServiceError
  ├── PipelineError
  └── ConfigurationError

✓ Exception hierarchy verified (15 classes, correct inheritance + status codes)


## 2. ErrorResponse Schema

In [3]:
from src.schemas.api.error import ErrorDetail, ErrorResponse

resp = ErrorResponse(error=ErrorDetail(
    type="PaperNotFoundError",
    message="Paper xyz not found",
    request_id="abc-123",
    detail={"paper_id": "xyz"},
))
data = resp.model_dump()
assert data["error"]["type"] == "PaperNotFoundError"
assert data["error"]["message"] == "Paper xyz not found"
assert data["error"]["request_id"] == "abc-123"
assert data["error"]["detail"] == {"paper_id": "xyz"}

import json
print(json.dumps(data, indent=2))
print("\n✓ ErrorResponse schema serializes correctly")

{
  "error": {
    "type": "PaperNotFoundError",
    "message": "Paper xyz not found",
    "request_id": "abc-123",
    "detail": {
      "paper_id": "xyz"
    }
  }
}

✓ ErrorResponse schema serializes correctly


## 3. Global Exception Handlers (via HTTP)

In [4]:
import httpx
from httpx import ASGITransport, AsyncClient
from src.main import create_app
from src.exceptions import PaperNotFoundError, ExternalServiceError, ArxivRateLimitError, PDFValidationError

app = create_app()

# Add test routes that raise specific exceptions
@app.get("/api/v1/test/not-found")
async def _raise_not_found():
    raise PaperNotFoundError("paper xyz not found")

@app.get("/api/v1/test/service-error")
async def _raise_service():
    raise ExternalServiceError("OpenSearch is down")

@app.get("/api/v1/test/rate-limit")
async def _raise_rate_limit():
    raise ArxivRateLimitError("429 Too Many Requests")

@app.get("/api/v1/test/validation")
async def _raise_validation():
    raise PDFValidationError("not a valid PDF")

@app.get("/api/v1/test/unhandled")
async def _raise_unhandled():
    raise RuntimeError("unexpected crash")

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    # 404
    r = await client.get("/api/v1/test/not-found")
    assert r.status_code == 404
    assert r.json()["error"]["type"] == "PaperNotFoundError"
    print(f"  PaperNotFoundError -> {r.status_code} {r.json()['error']['type']}")

    # 503
    r = await client.get("/api/v1/test/service-error")
    assert r.status_code == 503
    print(f"  ExternalServiceError -> {r.status_code} {r.json()['error']['type']}")

    # 429
    r = await client.get("/api/v1/test/rate-limit")
    assert r.status_code == 429
    print(f"  ArxivRateLimitError -> {r.status_code} {r.json()['error']['type']}")

    # 422
    r = await client.get("/api/v1/test/validation")
    assert r.status_code == 422
    print(f"  PDFValidationError -> {r.status_code} {r.json()['error']['type']}")

    # 500 (unhandled, no stack trace leak)
    r = await client.get("/api/v1/test/unhandled")
    assert r.status_code == 500
    assert r.json()["error"]["type"] == "InternalServerError"
    assert "Traceback" not in r.json()["error"]["message"]
    print(f"  RuntimeError (unhandled) -> {r.status_code} {r.json()['error']['type']}")

print("\n✓ All exception handlers return correct HTTP status codes and structured JSON")

GET /api/v1/test/not-found raised PaperNotFoundError: paper xyz not found [request_id=3dde5bdf-57b2-45cc-a92f-ce28ea2b1179]
GET /api/v1/test/service-error raised ExternalServiceError: OpenSearch is down [request_id=225e438c-546f-4e55-ac56-8822602f22a2]
GET /api/v1/test/rate-limit raised ArxivRateLimitError: 429 Too Many Requests [request_id=b86cd114-2880-4453-a0d5-79f36fd04c7c]
GET /api/v1/test/validation raised PDFValidationError: not a valid PDF [request_id=e397247b-60db-45a9-9290-2ffe88db0c0c]
GET /api/v1/test/unhandled raised unhandled RuntimeError: unexpected crash [request_id=efdd4e3c-ce11-4298-a75b-4cad4d9fe33e]
Traceback (most recent call last):
  File "/Users/nishantgaurav/Project/PaperAlchemy/notebooks/specs/../../src/middlewares.py", line 38, in dispatch
    response = await call_next(request)
               ^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/nishantgaurav/Project/PaperAlchemy/.venv/lib/python3.12/site-packages/starlette/middleware/base.py", line 168, in call_next
    r

  PaperNotFoundError -> 404 PaperNotFoundError
  ExternalServiceError -> 503 ExternalServiceError
  ArxivRateLimitError -> 429 ArxivRateLimitError
  PDFValidationError -> 422 PDFValidationError
  RuntimeError (unhandled) -> 500 InternalServerError

✓ All exception handlers return correct HTTP status codes and structured JSON


## 4. Request Logging Middleware (X-Request-ID + Timing)

In [5]:
async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    # X-Request-ID generated automatically
    r = await client.get("/api/v1/ping")
    req_id = r.headers.get("x-request-id")
    assert req_id is not None
    assert len(req_id) == 36  # UUID4 format
    print(f"  Auto-generated X-Request-ID: {req_id}")

    # X-Request-ID forwarded if provided
    custom_id = "my-custom-id-456"
    r = await client.get("/api/v1/ping", headers={"X-Request-ID": custom_id})
    assert r.headers["x-request-id"] == custom_id
    print(f"  Forwarded X-Request-ID: {r.headers['x-request-id']}")

    # Error responses also include request_id
    r = await client.get("/api/v1/test/not-found")
    assert r.json()["error"]["request_id"] is not None
    print(f"  Error response request_id: {r.json()['error']['request_id']}")

print("\n✓ Request logging middleware: X-Request-ID generated, forwarded, and included in error responses")

GET /api/v1/test/not-found raised PaperNotFoundError: paper xyz not found [request_id=1751efb8-b316-440b-b5ad-80ae00d73419]


  Auto-generated X-Request-ID: 15e05d93-4077-444e-8050-1136dcdf2335
  Forwarded X-Request-ID: my-custom-id-456
  Error response request_id: 1751efb8-b316-440b-b5ad-80ae00d73419

✓ Request logging middleware: X-Request-ID generated, forwarded, and included in error responses
